# Fresh original-sized value at risk

Native FlagQuantum qfunc, independent Circuit validation and actual count-only sampled bisection. Preserve source sizes and stopping semantics; the finite local estimator is not proprietary IQAE.


## Stage 1

Import public FlagQuantum and example-local fresh-model, search and plotting tools.


In [ ]:
import json
from dataclasses import asdict
from pathlib import Path

from IPython.display import display
from PIL import Image

from examples.synthesis.applications.finance.value_at_risk.overview.model import (
    discrete_quantile,
    fresh_model,
    search,
)
from examples.synthesis.applications.finance.value_at_risk.overview.plots import (
    plot_distribution,
)
from examples.synthesis.applications.finance.value_at_risk.overview.workflow import (
    ThresholdSampler,
)


## Stage 2

Compute the original seven-bit128point log-normal grid from fresh closed-form density.


In [ ]:
num_qubits = 7
mu, sigma = 0.7, 0.13
ALPHA, TOLERANCE = 0.07, 0.007
model = fresh_model(mu, sigma)
assert len(model.grid) == len(model.probabilities) == 2**num_qubits
print("Fresh model parameters:", mu, sigma, "points:", len(model.grid))


## Stage 3

Normalize fresh probabilities, compute the original discrete quantile and render all128points.


In [ ]:
assert abs(sum(model.probabilities) - 1) < 2e-15
quantile = discrete_quantile(model)
print("Fresh discrete7%quantile:", quantile)
figure_path = plot_distribution(model, Path("artifacts/development/value_at_risk"))
with Image.open(figure_path) as figure:
    display(figure.copy())
print(
    "Source legend says5%but its calculation uses7%; this figure labels the actual probability."
)


## Stage 4

Retain the source exclusive CDF sum(probs[:index]) as an independent classical comparison.


In [ ]:
print("P(asset index <32):", model.alpha(32))
assert model.alpha(0) == 0
assert abs(model.alpha(128) - 1) < 2e-15


## Stage 5

Retain the integer index update and halving schedule in the source search.


In [ ]:
initial_index = 2**num_qubits // 4
initial_search_size = initial_index // 2
print("Initial threshold:", initial_index, "search size:", initial_search_size)
assert (initial_index, initial_search_size) == (32, 16)


## Stage 6

Expose the unchanged bisection termination and the source implicit None return as diagnostics.


In [ ]:
print(
    "Stop on alpha tolerance or exhausted search size; no exact-CDF correction is applied."
)
print("The source function prints its internal index but returns None.")
print("Its first printed v is probs[index]; later values use grid_points[index].")


## Stage 7

Run the original exclusive-CDF classical bisection and record every source observation.


In [ ]:
classical = search(model, model.alpha)
for step in classical.steps:
    print(step)
print(
    "Classical internal index:", classical.index, "asset value:", classical.asset_value
)
print(
    "Original return:",
    classical.original_return,
    "tolerance reached:",
    classical.tolerance_reached,
)


## Stage 8

Create native distribution preparation, threshold XOR and reversible Grover family with independent full-state and clean-work validation.


In [ ]:
sampler = ThresholdSampler(model, seed=6200)
base = sampler.ensure(initial_index, 0)
print("Named outputs:", tuple(base.registers))
print(
    "Actual logical and clean-work resources under original width28:",
    base.report["resources"],
)
print("Independent absolute-state check:", sampler.checks[initial_index, 0])


## Stage 9

Bind each structural threshold and run the documented count-only local adaptive profile.


In [ ]:
print("Local amplitude profile: epsilon0.05,alpha0.01,k<=16,steps<=64,batch256.")
print("Fresh counts only enter estimation; exact CDFs are validation-only.")
print("Each new threshold/power compiles once; this is not runtime parameter binding.")
print(
    "Source search tolerance0.007is tighter than estimation precision0.05; convergence is not promised."
)


## Stage 10

Reset the original threshold before starting quantum-assisted bisection.


In [ ]:
index = 2**num_qubits // 4
assert index == initial_index == 32


## Stage 11

Run the entire sampled search, retain intervals/counts and report actual stopping behavior.


In [ ]:
measured = search(model, sampler.alpha_at)
for step in measured.steps:
    print(step)
for threshold, estimated in sampler.estimates:
    print("Threshold:", threshold, "sampled result:", estimated)
print("Measured internal index:", measured.index, "asset value:", measured.asset_value)
print(
    "Original return:",
    measured.original_return,
    "tolerance reached:",
    measured.tolerance_reached,
)
print("Private dropped asset is retained environment, not clean workspace.")
checks = tuple(sampler.checks.values())
assert all(check.state_error < 3e-11 and check.named_error < 3e-11 for check in checks)
assert measured.original_return is None
assert len(measured.steps) <= 6
print(
    "ARTIFACT_JSON="
    + json.dumps(
        {
            "points": len(model.grid),
            "seed": 6200,
            "classical": asdict(classical),
            "measured": asdict(measured),
            "estimates": [
                {"threshold": threshold, "result": asdict(result)}
                for threshold, result in sampler.estimates
            ],
            "program_count": len(sampler.programs),
            "checks": [asdict(check) for check in checks],
            "quantile": quantile,
            "actual_draws": sum(
                row.shots
                for _, result in sampler.estimates
                for row in result.iterations
            ),
        }
    )
)
